In [3]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from src.preprocessing import build_preprocessor
from src.models import get_model
from src.validation import evaluate_model

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


c:\Users\coffe\Desktop\titanic-ml\titanic-ml


In [4]:
raw_data = pd.read_csv('data/train.csv')

X = raw_data.drop(columns=['Survived'])
y = raw_data['Survived']

SEED = 42

In [29]:
model = get_model('logistic_regression', seed=SEED, max_iter=1000)
print(model)

pipeline = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', model),
])

result = evaluate_model(
    estimator=pipeline,
    X=X,
    y=y,
    seed=SEED,
    n_jobs=1
)

result

LogisticRegression(max_iter=1000, random_state=42)


{'scores': array([0.79888268, 0.82022472, 0.76966292, 0.79775281, 0.83707865]),
 'mean': np.float64(0.8047203565375683),
 'std': np.float64(0.022799532685402385)}

In [32]:
model = get_model(
    model_name='knn',
    #n_neighbors=7,
    #weights='uniform',
    #metric='minkowski',
)
print(model)

pipeline = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', model),
])

result = evaluate_model(
    estimator=pipeline,
    X=X,
    y=y,
    seed=SEED,
    n_jobs=1
)

result

KNeighborsClassifier()


{'scores': array([0.81005587, 0.79213483, 0.7752809 , 0.8258427 , 0.80898876]),
 'mean': np.float64(0.8024606113866047),
 'std': np.float64(0.017276487386580087)}

In [33]:
model = get_model(
    model_name='decision_tree'
)
print(model)

pipeline = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', model),
])

result = evaluate_model(
    estimator=pipeline,
    X=X,
    y=y,
    seed=SEED,
    n_jobs=1
)

result

DecisionTreeClassifier(random_state=42)


{'scores': array([0.79329609, 0.80898876, 0.75842697, 0.76966292, 0.78651685]),
 'mean': np.float64(0.7833783190006905),
 'std': np.float64(0.01774705384585448)}

In [36]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

tree_pipeline = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", get_model(
        "decision_tree",
        seed=SEED,
    )),
])

tree_pipeline.fit(X_train, y_train)

train_pred = tree_pipeline.predict(X_train)
val_pred = tree_pipeline.predict(X_val)

tree = tree_pipeline.named_steps["model"]

print("Tree depth:", tree.get_depth())
print("Number of leaves:", tree.get_n_leaves())

print(
    "Train accuracy:",
    accuracy_score(y_train, train_pred)
)

print(
    "Validation accuracy:",
    accuracy_score(y_val, val_pred)
)


Tree depth: 25
Number of leaves: 161
Train accuracy: 0.9859550561797753
Validation accuracy: 0.7821229050279329


In [37]:
depths = [2, 3, 4, 5, 6, 8, 10, 15, None]

for depth in depths:
    model = get_model(
        "decision_tree",
        seed=SEED,
        max_depth=depth,
    )

    pipeline = Pipeline([
        ("preprocessor", build_preprocessor()),
        ("model", model),
    ])

    result = evaluate_model(
        estimator=pipeline,
        X=X,
        y=y,
        seed=SEED,
        n_jobs=1,
    )

    print(
        f"max_depth={str(depth):>4} | "
        f"CV={result['mean']:.4f} ± {result['std']:.4f}"
    )

max_depth=   2 | CV=0.7778 ± 0.0106
max_depth=   3 | CV=0.8170 ± 0.0287
max_depth=   4 | CV=0.8148 ± 0.0229
max_depth=   5 | CV=0.8125 ± 0.0249
max_depth=   6 | CV=0.8182 ± 0.0127
max_depth=   8 | CV=0.8103 ± 0.0256
max_depth=  10 | CV=0.8047 ± 0.0252
max_depth=  15 | CV=0.7912 ± 0.0158
max_depth=None | CV=0.7834 ± 0.0177


In [38]:
leaf_sizes = [1, 2, 5, 10, 20, 30]

for leaf_size in leaf_sizes:
    model = get_model(
        "decision_tree",
        seed=SEED,
        max_depth=6,
        min_samples_leaf=leaf_size,
    )
    
    pipeline = Pipeline([
        ("preprocessor", build_preprocessor()),
        ("model", model),
    ])

    result = evaluate_model(
        estimator=pipeline,
        X=X,
        y=y,
        seed=SEED,
        n_jobs=1,
    )

    print(
        f"min_samples_leaf={str(leaf_size):>4} | "
        f"CV={result['mean']:.4f} ± {result['std']:.4f}"
    )

min_samples_leaf=   1 | CV=0.8182 ± 0.0127
min_samples_leaf=   2 | CV=0.8182 ± 0.0080
min_samples_leaf=   5 | CV=0.8249 ± 0.0141
min_samples_leaf=  10 | CV=0.8170 ± 0.0228
min_samples_leaf=  20 | CV=0.8002 ± 0.0277
min_samples_leaf=  30 | CV=0.8002 ± 0.0258


In [43]:
model = get_model(
    "decision_tree",
    seed=SEED,
)

pipeline = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", model),
])

In [44]:
param_grid = {
    'model__max_depth': [3, 4, 5, 6, 8, 10],
    'model__min_samples_leaf': [1, 2, 5, 10],
    'model__min_samples_split': [2, 5, 10, 20],
    'model__criterion': ['gini', 'entropy'],
}

In [46]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv,
    n_jobs=1,
    return_train_score=True
)

In [47]:
grid_search.fit(X, y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__criterion': ['gini', 'entropy'], 'model__max_depth': [3, 4, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_sco

In [48]:
print("Best parameters:")
print(grid_search.best_params_)

print(f"\nBest CV accuracy: {grid_search.best_score_:.4f}")

Best parameters:
{'model__criterion': 'gini', 'model__max_depth': 8, 'model__min_samples_leaf': 5, 'model__min_samples_split': 2}

Best CV accuracy: 0.8350


In [49]:
import pandas as pd

results = pd.DataFrame(grid_search.cv_results_)

results = results[
    [
        "param_model__max_depth",
        "param_model__min_samples_leaf",
        "param_model__min_samples_split",
        "param_model__criterion",
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ]
]

results.sort_values("rank_test_score").head(10)

,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,param_model__criterion,mean_train_score,mean_test_score,std_test_score,rank_test_score
74,8,5,10,gini,0.878503,0.835007,0.018404,1
72,8,5,2,gini,0.878503,0.835007,0.018404,1
73,8,5,5,gini,0.878503,0.835007,0.018404,1
89,10,5,5,gini,0.884679,0.831662,0.018344,4
88,10,5,2,gini,0.884679,0.831662,0.018344,4
90,10,5,10,gini,0.884679,0.831662,0.018344,4
185,10,5,5,entropy,0.885801,0.828291,0.016802,7
184,10,5,2,entropy,0.885801,0.828291,0.016802,7
186,10,5,10,entropy,0.885801,0.828291,0.016802,7
91,10,5,20,gini,0.870649,0.828266,0.025378,10


In [ ]:
# random_forest baseline

model = get_model(
    model_name='random_forest',
    seed=SEED,
)

pipeline = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', model),
])

result = evaluate_model(
    estimator=pipeline,
    X=X,
    y=y,
    seed=SEED,
    n_jobs=1
)

result

{'scores': array([0.83798883, 0.80898876, 0.79213483, 0.82022472, 0.82022472]),
 'mean': np.float64(0.8159123721047015),
 'std': np.float64(0.015085558083696077)}

In [6]:
n_estimators_values = [1, 5, 10, 25, 50, 100, 200, 500]

for n in n_estimators_values:
    model = get_model(
        "random_forest",
        seed=SEED,
        n_estimators=n,
    )

    pipeline = Pipeline([
        ("preprocessor", build_preprocessor()),
        ("model", model),
    ])

    result = evaluate_model(
        estimator=pipeline,
        X=X,
        y=y,
        seed=SEED,
        n_jobs=1,
    )

    print(
        f"n_estimators={n:>3} | "
        f"CV={result['mean']:.4f} ± {result['std']:.4f}"
    )

n_estimators=  1 | CV=0.7485 ± 0.0450
n_estimators=  5 | CV=0.7980 ± 0.0219
n_estimators= 10 | CV=0.8047 ± 0.0234
n_estimators= 25 | CV=0.8159 ± 0.0247
n_estimators= 50 | CV=0.8148 ± 0.0183
n_estimators=100 | CV=0.8159 ± 0.0151
n_estimators=200 | CV=0.8148 ± 0.0149
n_estimators=500 | CV=0.8182 ± 0.0138


In [7]:
max_features_values = [
    1,
    2,
    3,
    "sqrt",
    "log2",
    0.5,
    None,
]

for max_features in max_features_values:
    model = get_model(
        "random_forest",
        seed=SEED,
        n_estimators=200,
        max_features=max_features,
    )

    pipeline = Pipeline([
        ("preprocessor", build_preprocessor()),
        ("model", model),
    ])

    result = evaluate_model(
        estimator=pipeline,
        X=X,
        y=y,
        seed=SEED,
        n_jobs=1,
    )

    print(
        f"max_features={str(max_features):>4} | "
        f"CV={result['mean']:.4f} ± {result['std']:.4f}"
    )

max_features=   1 | CV=0.8182 ± 0.0195
max_features=   2 | CV=0.8125 ± 0.0220
max_features=   3 | CV=0.8148 ± 0.0149
max_features=sqrt | CV=0.8148 ± 0.0149
max_features=log2 | CV=0.8148 ± 0.0149
max_features= 0.5 | CV=0.8249 ± 0.0211
max_features=None | CV=0.8193 ± 0.0234


In [10]:
model = get_model(
    model_name='random_forest',
    seed=SEED,
    n_estimators=200
)

pipeline = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', model),
])

param_grid = {
    "model__max_depth": [4, 6, 8, 10, None],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__max_features": ["sqrt", 0.5, None],
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv,
    n_jobs=1,
    return_train_score=True
)

grid_search.fit(X, y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [4, 6, ...], 'model__max_features': ['sqrt', 0.5, ...], 'model__min_samples_leaf': [1, 2, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be

In [11]:
print("Best parameters:")
print(grid_search.best_params_)

print(f"\nBest CV accuracy: {grid_search.best_score_:.4f}")

Best parameters:
{'model__max_depth': 10, 'model__max_features': 0.5, 'model__min_samples_leaf': 2}

Best CV accuracy: 0.8372


In [14]:
results = pd.DataFrame(grid_search.cv_results_)

results = results[
    [
        "param_model__max_depth",
        "param_model__min_samples_leaf",
        "param_model__max_features",
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ]
].sort_values("rank_test_score")

results.head(10)

,param_model__max_depth,param_model__min_samples_leaf,param_model__max_features,mean_train_score,mean_test_score,std_test_score,rank_test_score
41,10,2,0.5,0.914141,0.837236,0.014892,1
33,8,2,None,0.912739,0.836112,0.025373,2
57,None,2,None,0.931817,0.836112,0.019781,2
25,8,2,sqrt,0.892816,0.835001,0.016271,4
21,6,2,None,0.892255,0.835001,0.024091,4
45,10,2,None,0.922839,0.834995,0.021349,6
32,8,1,None,0.937710,0.834995,0.022501,6
49,None,2,sqrt,0.910773,0.834988,0.020480,8
20,6,1,None,0.898428,0.833871,0.029967,9
44,10,1,None,0.958472,0.833865,0.023901,10


In [15]:
grid_search.best_params_

{'model__max_depth': 10,
 'model__max_features': 0.5,
 'model__min_samples_leaf': 2}